# Titans Integration: Architectural Analysis of Qwen-2.5

This notebook documents the reverse-engineering and architectural mapping of the Qwen-2.5-1.5B-Instruct model. The goal is to identify the precise injection points for the Titans Neural Memory module.

In [ ]:
!uv pip install -q transformers torch titans-pytorch accelerate datasets

: 

In [6]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from titans_pytorch import NeuralMemory

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

Transformers version: 4.57.6
PyTorch version: 2.8.0
Device: mps


### Model Initialization

We initialize Qwen-2.5-1.5B-Instruct. We extract the config object, which serves as the architectural blueprint for our injection.

In [7]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

config = model.config
print(f"Architecture: {config.architectures[0]}")
print(f"Number of Layers: {config.num_hidden_layers}")
print(f"Hidden Size (d_model): {config.hidden_size}")
print(f"Intermediate Size (MLP): {config.intermediate_size}")

Some parameters are on the meta device because they were offloaded to the disk.


Architecture: Qwen2ForCausalLM
Number of Layers: 28
Hidden Size (d_model): 1536
Intermediate Size (MLP): 8960


To successfully replace or augment the Attention layers with Titans, we must visualize the internal naming of the sub-modules. We are specifically looking for the Qwen2DecoderLayer and its internal self_attn component.

In [3]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

### Analysis of Attention Projections (GQA)

From the model summary, we observe that Qwen-2.5-1.5B:
- implements Grouped Query Attention (GQA).
- the hidden_size is 1536.
- the q_proj maintains the full dimensionality of 1536
- the k_proj and v_proj are down-projected to 256.

To calculate the number of heads, we divide the output features by the head dimension: 
- $1536 / 128 = 12$ Query Heads
- $256 / 128 = 2$ KV Heads

Implication for Titans: The NeuralMemory module from titans-pytorch typically expects a fixed dimension. We must decide whether to place Titans before the projections or replace the projections entirely.

In [4]:
# Programmatic extraction of dimensions
layer_zero = model.model.layers[0].self_attn

d_model = config.hidden_size
n_heads = config.num_attention_heads
n_kv_heads = config.num_key_value_heads
head_dim = d_model // n_heads

print(f"--- Architectural Parameters ---")
print(f"🔹 Hidden Dimension (d_model): {d_model}")
print(f"🔹 Number of Heads (Q): {n_heads}")
print(f"🔹 Number of KV Heads (GQA): {n_kv_heads}")
print(f"🔹 Dimension per Head: {head_dim}")
print(f"🔹 MLP Intermediate Size: {config.intermediate_size}")

--- Architectural Parameters ---
🔹 Hidden Dimension (d_model): 1536
🔹 Number of Heads (Q): 12
🔹 Number of KV Heads (GQA): 2
🔹 Dimension per Head: 128
🔹 MLP Intermediate Size: 8960


### Identifying the Weight Tensors
Before injecting Titans, we need to inspect the weights of a specific layer. This is crucial if we plan to perform Partial Weight Transfer (reusing Qwen's pre-trained knowledge to initialize the Titans gate or linear layers).

We will extract the shape of the projection weights for Layer 0.

In [5]:
# Inspecting weight shapes for the first layer
print(f"Layer 0 - Query Projection Weight Shape: {layer_zero.q_proj.weight.shape}")
print(f"Layer 0 - KV Projection Weight Shape: {layer_zero.k_proj.weight.shape}")
print(f"Layer 0 - Output Projection Weight Shape: {layer_zero.o_proj.weight.shape}")

# Verify if bias is present (important for custom layer implementation)
print(f"Query Projection has bias: {layer_zero.q_proj.bias is not None}")

Layer 0 - Query Projection Weight Shape: torch.Size([1536, 1536])
Layer 0 - KV Projection Weight Shape: torch.Size([256, 1536])
Layer 0 - Output Projection Weight Shape: torch.Size([1536, 1536])
Query Projection has bias: True


### Titans Neural Memory Mockup
Using the titans-pytorch library, we can now define a draft of the NeuralMemory module using the dimensions extracted above.

There are two main integration strategies:
- Replacement: Replacing the Qwen2Attention class entirely with a NeuralMemory block.
- Memory Augmentation: Keeping the Attention but adding a NeuralMemory as a "Long-term" branch.

In [17]:
# Mocking a Titans Neural Memory layer with Qwen's dimensions
titans_memory = NeuralMemory(
    dim = d_model,             # 1536
    heads = n_heads,           # 12
    dim_head = head_dim,       # 128
    chunk_size = 64,           # Standard for Titans
    # We will define depth and other hyperparameters in the final implementation
)

print("✅ Titans Neural Memory successfully initialized with Qwen dimensions.")
print(titans_memory)

✅ Titans Neural Memory successfully initialized with Qwen dimensions.
NeuralMemory(
  (assoc_scan): AssocScan()
  (retrieve_norm): RMSNorm((1536,), eps=None, elementwise_affine=True)
  (store_norm): RMSNorm((1536,), eps=None, elementwise_affine=True)
  (multihead_rmsnorm): Identity()
  (q_norm): Identity()
  (k_norm): Identity()
  (split_heads): Rearrange('b n (h d) -> b h n d', h=12)
  (split_kv_heads): Rearrange('b n (h u d) -> b h (n u) d', h=12, u=1)
  (merge_heads): Rearrange('b h n d -> b n (h d)')
  (combine_heads): Linear(in_features=1536, out_features=1536, bias=False)
  (retrieve_gate): Sequential(
    (0): Linear(in_features=1536, out_features=12, bias=False)
    (1): Rearrange('b n h -> b h n 1')
    (2): Sigmoid()
  )
  (memory_model): ResidualNorm(
    (norm): LayerNorm(
      (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=False)
    )
    (model): MemoryMLP(
      (weights): ParameterList(
          (0): Parameter containing: [torch.float32 of size 128x512]
      